In [ ]:
# Imports and configuration
import os
import math
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

# Backend base URL (set via CASE_MGMT_BACKEND_URL)
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3000')

# Fallback account ID (used when no account specified or when fetch returns 404)
FALLBACK_ACCOUNT_ID = 'dbtrAcct_de52b2c4041c4a03a73a92cfb0b8cf35'

# Account to visualize can be provided via env var ACCOUNT_ID, otherwise fallback used
accountId = os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)

# print(f'Using backend: {backendUrl}')
# print(f'Using accountId: {accountId}')


In [ ]:
# Fetch network analysis data with fallback on 404
def fetch_account_network(account_id, backend_url):
    url = f"{backend_url}/api/v1/lakehouse/network-analysis/account/{account_id}"
    params = {'tenantId': 'DEFAULT'}
    try:
        resp = requests.get(url, params=params, timeout=15)
    except Exception as e:
        print('Request error:', e)
        return None

    # If not found and we're not already using the fallback, try fallback
    if resp.status_code == 404 and account_id != FALLBACK_ACCOUNT_ID:
        print(f'Network data not found for {account_id}, trying fallback: {FALLBACK_ACCOUNT_ID}')
        url = f"{backend_url}/api/v1/lakehouse/network-analysis/account/{FALLBACK_ACCOUNT_ID}"
        try:
            resp = requests.get(url, params=params, timeout=15)
        except Exception as e:
            print('Request error (fallback):', e)
            return None

    try:
        resp.raise_for_status()
    except Exception as e:
        print('HTTP error:', e)
        return None

    return resp.json()

# Example: fetch data now
data = fetch_account_network(accountId, backendUrl)
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {accountId}</div>"))
# else:
#     print('Fetched network data keys:', list(data.keys()))


In [ ]:
# Normalise expected network payloads and build node/edge lists
# Actual format from backend: { 'network': { 'nodes': [...], 'edges': [...] }, 'accountDetails': {...}, 'meta': {...} }

nodes = []
edges = []

if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        nodes = network_data.get('nodes', [])
        edges = network_data.get('edges', [])
    else:
        # Fallback to direct keys if structure differs
        nodes = data.get('nodes', [])
        edges = data.get('edges', [])

# Ensure simple structures
nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]

# print(f'Nodes: {len(nodes)}, Edges: {len(edges)}')
# if nodes:
#     print('Sample node:', nodes[0])
# if edges:
#     print('Sample edge:', edges[0])

# Build a simple layout (circular + center emphasis) and render with Plotly
def build_positions(nodes):
    positions = {}
    n = max(len(nodes), 1)
    # find center node (isCenter flag or the fallback account id)
    center_idx = 0
    for i, node in enumerate(nodes):
        if node.get('isCenter') or node.get('id') == accountId or node.get('id') == FALLBACK_ACCOUNT_ID:
            center_idx = i
            break

    # Place center at origin, others on circle
    radius = 300
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes)

# Create Plotly traces for edges (lines) and nodes (scatter)
edge_x = []
edge_y = []
for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#cbd5e1'), hoverinfo='none')

# Node scatter - map types/status to marker styles
node_x = []
node_y = []
node_text = []
marker_size = []
marker_color = []
node_ids = []

def color_for(node):
    status = node.get('status', 'normal')
    t = node.get('type', 'account')
    if status in ('alert', 'flagged'):
        return '#ef4444' # Red-500
    if status == 'investigation':
        return '#f59e0b' # Amber-500
    if t == 'counterparty':
        return '#6366f1' # Indigo-500
    return '#3b82f6' # Blue-500

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue
    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)
    label = n.get('label') or nid
    node_text.append(f"{label} ({nid})")
    marker_size.append(32 if n.get('isCenter') else 22)
    marker_color.append(color_for(n))
    node_ids.append(nid)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=[t.split(' ')[0] for t in node_text],
    hovertext=node_text,
    hoverinfo='text',
    customdata=node_ids,
    marker=dict(
        showscale=False,
        color=marker_color,
        size=marker_size,
        line_width=2,
        line_color='#ffffff'
    ),
    textposition='bottom center'
)

fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
    title=None,
    showlegend=False,
    hovermode='closest',
    margin=dict(b=20,l=5,r=5,t=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
))

if nodes:
    import json
    graph_id = 'network-graph-id'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    # Sidebar HTML with dynamic placeholders
    sidebar_html = """
    <div id="sidebar-container" style="width: 380px; background: #ffffff; color: #0f172a; padding: 24px; border-radius: 24px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); font-family: 'Inter', system-ui, sans-serif; margin-left: 24px; border: 1px solid #e2e8f0; display: flex; flex-direction: column; height: 600px; overflow-y: auto; transition: all 0.3s ease;">
        <!-- OVERVIEW STATE -->
        <div id="view-overview">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Counterparty Details</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Counterparty ID</div>
                <div id="overview-id" style="font-weight: 600; font-size: 1rem; color: #0f172a;">-</div>
            </div>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Name</div>
                <div id="overview-name" style="font-weight: 600; font-size: 1.1rem; color: #0284c7;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Type</div>
                <div id="overview-type" style="display: inline-block; background: #f1f5f9; color: #475569; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">-</div>
            </div>

            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Network Summary</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Linked Accounts</span>
                        <span id="overview-linked" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Transactions</span>
                        <span id="overview-total-tx" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Value</span>
                        <span id="overview-total-value" style="font-weight: 600; color: #059669;">$0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Accounts with Alerts</span>
                        <span id="overview-alerts" style="font-weight: 600; color: #dc2626;">0</span>
                    </div>
                </div>
            </div>
        </div>

        <!-- SELECTED STATE -->
        <div id="view-details" style="display: none;">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Account Details</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Account ID</div>
                <div id="side-id" style="font-weight: 600; font-size: 1rem; color: #0f172a; word-break: break-all;">-</div>
            </div>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Account Holder</div>
                <div id="side-name" style="font-weight: 600; font-size: 1.1rem; color: #d97706;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Relationship</div>
                <div id="side-relation" style="display: inline-block; background: #fff7ed; color: #9a3412; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">Primary Owner</div>
            </div>

            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Transaction Statistics</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Transactions</span>
                        <span id="side-tx" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Value</span>
                        <span id="side-value" style="font-weight: 600; color: #059669;">$0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Velocity</span>
                        <span id="side-velocity" style="font-weight: 600; color: #dc2626;">HIGH</span>
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 24px; text-align: center;">
                <button onclick="resetSidebar()" style="background: white; border: 1px solid #cbd5e1; color: #64748b; padding: 8px 16px; border-radius: 8px; font-size: 0.75rem; cursor: pointer; font-weight: 500; transition: all 0.2s;">Back to Overview</button>
            </div>
        </div>
        
        <div style="margin-top: auto; padding-top: 20px; font-size: 0.65rem; color: #475569; text-align: center;">
            Click a node for details • Click background to reset
        </div>
    </div>
    """
    
    # Interactivity Script
    account_details_json = json.dumps(data.get('accountDetails', {}))
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const accountDetails = {account_details_json};
        const graphDiv = document.getElementById('{graph_id}');
        
        function initializeOverview() {{
            document.getElementById('overview-id').innerText = accountDetails.accountId || '-';
            document.getElementById('overview-name').innerText = accountDetails.accountHolder || '-';
            document.getElementById('overview-type').innerText = 'Business Account';
            
            const linkedAccountsCount = nodes.length - 1; // All nodes except root
            let totalTxCount = 0;
            let totalValueAmount = 0;
            let totalAlertsCount = 0;
            
            edges.forEach(e => {{
                totalTxCount += e.txCount || 0;
                totalValueAmount += e.totalAmount || 0;
            }});
            
            nodes.forEach(n => {{
                if (n.status === 'alert' || n.status === 'flagged' || (n.flags && n.flags.alerted)) {{
                    totalAlertsCount++;
                }}
            }});
            
            // Special case: check accountDetails itself for alert
            if (accountDetails.flags && accountDetails.flags.alerted) {{
                // Ensure we don't double count if it's already in the nodes list
                // But usually root is in nodes
            }}
            
            document.getElementById('overview-linked').innerText = Math.max(linkedAccountsCount, 0);
            document.getElementById('overview-total-tx').innerText = totalTxCount;
            document.getElementById('overview-total-value').innerText = '$' + totalValueAmount.toLocaleString(undefined, {{minimumFractionDigits: 2}});
            document.getElementById('overview-alerts').innerText = totalAlertsCount;
        }}
        
        window.resetSidebar = function() {{
            document.getElementById('view-overview').style.display = 'block';
            document.getElementById('view-details').style.display = 'none';
            document.getElementById('sidebar-container').style.background = '#ffffff';
            initializeOverview();
        }};
        
        function updateSidebar(nodeId) {{
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            
            document.getElementById('view-overview').style.display = 'none';
            document.getElementById('view-details').style.display = 'block';
            document.getElementById('sidebar-container').style.background = '#ffffff';
            
            const linkedEdges = edges.filter(e => 
                (e.source || e.from) === nodeId || 
                (e.target || e.to) === nodeId
            );
            
            let txCount = 0;
            let totalVal = 0;
            let velocity = 'MEDIUM';
            let holder = node.label || node.id;
            let relation = 'Counterparty';
            
            if (nodeId === accountDetails.accountId) {{
                holder = accountDetails.accountHolder || holder;
                txCount = accountDetails.transactions || 0;
                totalVal = accountDetails.totalValue || 0;
                velocity = accountDetails.velocity || 'LOW';
                relation = accountDetails.relationship || 'Primary Owner';
            }} else {{
                linkedEdges.forEach(e => {{
                    txCount += e.txCount || 0;
                    totalVal += e.totalAmount || 0;
                }});
                velocity = txCount > 10 ? 'HIGH' : 'LOW';
            }}

            document.getElementById('side-id').innerText = node.id;
            document.getElementById('side-name').innerText = holder;
            document.getElementById('side-relation').innerText = relation;
            document.getElementById('side-tx').innerText = txCount;
            document.getElementById('side-value').innerText = '$' + totalVal.toLocaleString(undefined, {{minimumFractionDigits: 2}});
            document.getElementById('side-velocity').innerText = velocity;
            
            const container = document.getElementById('sidebar-container');
            container.style.transform = 'scale(0.98)';
            setTimeout(() => {{ container.style.transform = 'scale(1)'; }}, 100);
        }}
        
        // Initial setup
        setTimeout(() => initializeOverview(), 500);
        
        if (graphDiv) {{
            graphDiv.on('plotly_click', function(data) {{
                if (data.points && data.points[0] && data.points[0].customdata) {{
                    updateSidebar(data.points[0].customdata);
                }}
            }});
            
            graphDiv.on('plotly_doubleclick', resetSidebar);
        }}
    }})();
    </script>
    """
    
    container_html = f"""
    <div style="display: flex; align-items: stretch; background-color: #ffffff; padding: 24px; border-radius: 32px; gap: 24px; border: 1px solid #f1f5f9;">
        <div style="flex: 1; background: #f8fafc; border-radius: 24px; border: 1px solid #e2e8f0; overflow: hidden; position: relative;">
            {graph_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    
    display(HTML(container_html))
else:
    display(HTML('<div>No nodes to display</div>'))
